# Build intermediary candidates from Korea Customs API

Output: `data/interim/intermediary_candidates.csv` with one row per `event_id`.

Columns: `event_id`, `????`, `hs_code`, `???`, `?????`.

Collection window: tariff start date ? 1 year.


In [ ]:
from __future__ import annotations

import sys
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "trade-circumvention-monitor" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_customs_api_key

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
REGULATION_EVENTS_PATH = DATA_INTERIM / "regulation_events.csv"
PLAN_PATH = DATA_INTERIM / "intermediary_candidates_collection_plan.csv"
STATUS_PATH = DATA_INTERIM / "intermediary_candidates_collection_status.csv"
RAW_PATH = DATA_INTERIM / "intermediary_candidates_customs_raw.csv"
OUTPUT_PATH = DATA_INTERIM / "intermediary_candidates.csv"

CUSTOMS_URL = "https://apis.data.go.kr/1220000/nitemtrade/getNitemtradeList"
TODAY = pd.Timestamp("2026-05-17")
CHECKPOINT_EVERY = 25


In [ ]:
def split_hs_codes(value: object) -> list[str]:
    if pd.isna(value):
        return []
    text = str(value).replace(",", ";")
    return [part.strip() for part in text.split(";") if part.strip()]


def collection_window(start_date: object, today: pd.Timestamp = TODAY) -> tuple[pd.Timestamp, pd.Timestamp]:
    duty_start = pd.to_datetime(start_date)
    start = duty_start - pd.DateOffset(years=1)
    end = min(duty_start + pd.DateOffset(years=1), today)
    return start.normalize(), end.normalize()


def split_12_month_chunks(start: pd.Timestamp, end: pd.Timestamp) -> list[tuple[str, str]]:
    start_period = pd.Period(start, freq="M")
    end_period = pd.Period(end, freq="M")
    chunks = []
    current = start_period
    while current <= end_period:
        chunk_end = min(current + 11, end_period)
        chunks.append((current.strftime("%Y%m"), chunk_end.strftime("%Y%m")))
        current = chunk_end + 1
    return chunks


def safe_to_csv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_csv(path, index=False, encoding="utf-8-sig")
        return path
    except PermissionError:
        fallback = path.with_name(f"{path.stem}_latest{path.suffix}")
        df.to_csv(fallback, index=False, encoding="utf-8-sig")
        return fallback


def read_csv_or_empty(path: Path, columns: list[str]) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path, dtype=str).fillna("")
    return pd.DataFrame(columns=columns)


def join_unique(values: pd.Series) -> str:
    cleaned = sorted({str(v).strip() for v in values if str(v).strip() and str(v).strip() != "-"})
    return ",".join(cleaned)


In [ ]:
def make_collection_plan(events: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, event in events.iterrows():
        hs_codes = split_hs_codes(event.get("hs_code"))
        if not hs_codes:
            continue
        start, end = collection_window(event["start_date"])
        for hs_code in hs_codes:
            for chunk_start, chunk_end in split_12_month_chunks(start, end):
                rows.append({
                    "request_id": f"REQ-{len(rows) + 1:08d}",
                    "event_id": event["event_id"],
                    "????": f"{start.strftime("%Y-%m-%d")}~{end.strftime("%Y-%m-%d")}",
                    "hs_code": hs_code,
                    "original_hs_code": event["hs_code"],
                    "???": event["origin_country_name_kr"],
                    "regulated_iso3": event.get("origin_country_iso3", ""),
                    "start_yymm": chunk_start,
                    "end_yymm": chunk_end,
                })
    return pd.DataFrame(rows)


events = pd.read_csv(REGULATION_EVENTS_PATH, dtype=str).fillna("")
plan = make_collection_plan(events)
safe_to_csv(plan, PLAN_PATH)
print("events", events.shape)
print("plan", plan.shape)
plan.head()


In [ ]:
def parse_customs_xml(text: str) -> tuple[str | None, str | None, list[dict]]:
    root = ET.fromstring(text)
    result_code = root.findtext("./header/resultCode")
    result_msg = root.findtext("./header/resultMsg")
    rows = []
    for item in root.findall("./body/items/item"):
        year = item.findtext("year") or ""
        country = item.findtext("statCdCntnKor1") or ""
        country_code = item.findtext("statCd") or ""
        if year == "??":
            continue
        rows.append({
            "year": year,
            "intermediary_country": country,
            "intermediary_customs_code": country_code,
            "response_hs_code": item.findtext("hsCd") or "",
            "imp_dlr": int(item.findtext("impDlr") or 0),
            "imp_wgt": int(item.findtext("impWgt") or 0),
        })
    return result_code, result_msg, rows


def collect_one_request(request: pd.Series, service_key: str) -> list[dict]:
    params = {
        "serviceKey": service_key,
        "strtYymm": request["start_yymm"],
        "endYymm": request["end_yymm"],
        "hsSgn": request["hs_code"],
    }
    response = requests.get(CUSTOMS_URL, params=params, timeout=30)
    response.raise_for_status()
    result_code, result_msg, rows = parse_customs_xml(response.text)
    if result_code != "00":
        raise RuntimeError(f"Customs API error {result_code}: {result_msg}")
    return [{**request.to_dict(), **row} for row in rows]


In [ ]:
service_key = get_customs_api_key()
status_columns = ["request_id", "status", "rows", "error", "started_at", "finished_at"]
raw_columns = list(plan.columns) + [
    "year",
    "intermediary_country",
    "intermediary_customs_code",
    "response_hs_code",
    "imp_dlr",
    "imp_wgt",
]

status = read_csv_or_empty(STATUS_PATH, status_columns)
raw = read_csv_or_empty(RAW_PATH, raw_columns)
done = set(status.loc[status["status"].eq("ok"), "request_id"])
status_records = status.to_dict("records")
raw_records = raw.to_dict("records")

for idx, request in plan.iterrows():
    if request["request_id"] in done:
        continue
    started_at = pd.Timestamp.now().isoformat(timespec="seconds")
    try:
        rows = collect_one_request(request, service_key)
        raw_records.extend(rows)
        status_records.append({
            "request_id": request["request_id"],
            "status": "ok",
            "rows": str(len(rows)),
            "error": "",
            "started_at": started_at,
            "finished_at": pd.Timestamp.now().isoformat(timespec="seconds"),
        })
        done.add(request["request_id"])
    except Exception as exc:  # noqa: BLE001
        status_records.append({
            "request_id": request["request_id"],
            "status": "error",
            "rows": "0",
            "error": repr(exc)[:500],
            "started_at": started_at,
            "finished_at": pd.Timestamp.now().isoformat(timespec="seconds"),
        })

    if (idx + 1) % CHECKPOINT_EVERY == 0 or idx == len(plan) - 1:
        saved_status = safe_to_csv(pd.DataFrame(status_records, columns=status_columns), STATUS_PATH)
        saved_raw = safe_to_csv(pd.DataFrame(raw_records, columns=raw_columns), RAW_PATH)
        print(f"checkpoint {idx + 1}/{len(plan)} done={len(done)} raw_rows={len(raw_records)} status={saved_status.name} raw={saved_raw.name}")
        time.sleep(0.2)

safe_to_csv(pd.DataFrame(status_records, columns=status_columns), STATUS_PATH)
safe_to_csv(pd.DataFrame(raw_records, columns=raw_columns), RAW_PATH)
print("done", len(done), "of", len(plan), "raw_rows", len(raw_records))


In [ ]:
raw = pd.read_csv(RAW_PATH, dtype=str).fillna("")
plan = pd.read_csv(PLAN_PATH, dtype=str).fillna("")

period_col = plan.columns[2]
regulated_col = plan.columns[5]
raw_regulated_col = raw.columns[5]

for col in ["imp_dlr", "imp_wgt"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce").fillna(0)

excluded_country_names = {
    "", "-", "\ucd1d\uacc4", "\uc804\uccb4", "\uc804\uad6d", "\uc138\uacc4",
    "World", "WORLD", "All", "ALL", "TOTAL", "Total",
}
excluded_country_codes = {"", "-", "0", "TOTAL", "Total", "WORLD", "World", "ALL", "All", "KR"}

valid = raw[
    ~raw["intermediary_country"].isin(excluded_country_names)
    & ~raw["intermediary_customs_code"].isin(excluded_country_codes)
    & raw["intermediary_country"].ne(raw[raw_regulated_col])
    & ((raw["imp_dlr"] > 0) | (raw["imp_wgt"] > 0))
].copy()

candidate_summary = (
    valid.groupby("event_id", dropna=False)
    .agg(candidate=("intermediary_country", join_unique))
    .reset_index()
)

event_meta = plan[["event_id", period_col, "original_hs_code", regulated_col]].drop_duplicates("event_id")
event_meta = event_meta.rename(
    columns={period_col: "period", "original_hs_code": "hs_code", regulated_col: "regulated"}
)
output = event_meta.merge(candidate_summary, on="event_id", how="left")
output["candidate"] = output["candidate"].fillna("")
output = output[["event_id", "period", "hs_code", "regulated", "candidate"]]
output.columns = ["event_id", "\uc0ac\uac74\uae30\uac04", "hs_code", "\uaddc\uc81c\uad6d", "\uc911\uac04\uad6d\ud6c4\ubcf4"]
safe_to_csv(output, OUTPUT_PATH)

print("saved", OUTPUT_PATH)
print("rows", len(output))
print("events_with_candidates", output["\uc911\uac04\uad6d\ud6c4\ubcf4"].ne("").sum())
print("events_without_candidates", output["\uc911\uac04\uad6d\ud6c4\ubcf4"].eq("").sum())
output.head(20)
